# 01 - Exploratory Data Analysis (EDA)

**Purpose:** before any deeper analysis, validate that the two source tables are
fit for the **TOBi routing & escalation** business question. Per data-science
best practice this notebook explores the data, surfaces quality issues, and
documents the assumptions on which the rest of the work depends.

**Audience:** data-science lead + junior data-scientist colleagues. The notebook
is designed to be readable top-to-bottom - every section has a **Why** note up
front and a **Key insight** under each result.

**Scope (the two source tables):**
- `vf-pt-copsvertex-live.vfpt_dh_lake_cops_pub_investigation.f_kafka_tobi_sessions` - one row per session
- `vf-pt-copsvertex-live.vfpt_dh_lake_cops_pub_investigation.f_tobi_logs_vertex` - one row per log token within a session

**Order of work:** *EDA -> Cleaning -> Analysis.* This notebook is the EDA.
Cleaning rules derived from these findings are documented in
[`docs/data_cleaning.md`](../docs/data_cleaning.md) and applied in
[`standalone/session_master_query.sql`](../standalone/session_master_query.sql).
Analysis lives in [`02_executive_dashboard.ipynb`](02_executive_dashboard.ipynb).

## TL;DR - what the EDA found

1. **The two tables join cleanly** on `SESSION_ID`; nearly all sessions have log events.
2. **Date quality issue** - raw data contains `null` and `1900-01-01` rows.
   *Decision:* apply a clean analysis window (`2024-01-01` to `2025-12-31`).
3. **`CONFIDENCE_LEVEL`** is a numeric score stored as **string**, dominated by `0`/blank.
   *Decision:* bucket into bands; do not rely on it as a primary driver.
4. **`IS_FUNCTIONAL`** values are `Yes` / `No` / blank / null.
   *Decision:* treat `Yes` as functional/contained for FCR.
5. **`T_` tags follow the published grammar** `T_<digit><letter><roman>_<client>`.
   *Decision:* decode the digit (1=Contained, 2=Transferred), letter (channel/outcome),
   Roman numeral (I=Non-tech / II=Tech / III=Commercial) and client-type suffix.
6. **Technical topic detection** uses entity codes (`E#`) from `S_` tokens - more
   accurate than keyword matching.

**Open items for the data-science lead to confirm** are flagged in Section 13.

## 1 - Setup & connect

**Why:** establish a reproducible BigQuery connection and auto-detect the source
region so subsequent queries don't fail with cross-region errors.

In [ ]:
# %pip install google-cloud-bigquery db-dtypes pandas matplotlib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from google.cloud import bigquery

PROJECT = 'vf-pt-copsvertex-live'
SOURCE  = f'{PROJECT}.vfpt_dh_lake_cops_pub_investigation'
SES = f'`{SOURCE}.f_kafka_tobi_sessions`'
LOG = f'`{SOURCE}.f_tobi_logs_vertex`'

client = bigquery.Client(project=PROJECT)
LOCATION = client.get_dataset(SOURCE).location
print('source region =', LOCATION)
def q(sql): return client.query(sql, location=LOCATION).to_dataframe()

pd.set_option('display.max_columns', 60)
plt.rcParams.update({'figure.dpi':110, 'axes.grid':True, 'grid.alpha':0.3})

## 2 - Sessions table: volume & coverage

**Why:** confirm the dataset is the size we expect for the time window and that
the timestamps are sane before computing anything else.

In [ ]:
q(f'''
SELECT COUNT(*)                                                       AS total_rows,
       COUNT(DISTINCT SESSION_ID)                                     AS distinct_sessions,
       MIN(START_MOMENT)                                              AS earliest_session,
       MAX(END_MOMENT)                                                AS latest_session,
       COUNTIF(NEXT_SESSION_ID IS NOT NULL AND NEXT_SESSION_ID != '')      AS sessions_with_next,
       COUNTIF(INTERNAL_SES_LIST IS NOT NULL AND INTERNAL_SES_LIST != '') AS sessions_with_internal
FROM {SES}''')

**Key insight:** the session table is large and `SESSION_ID` is effectively the
unique row key. `NEXT_SESSION_ID` and `INTERNAL_SES_LIST` are present on a notable
share of sessions - these are how customers re-contact / get handed over, so they
feed the misroute-impact metric later.

## 3 - Logs table: volume & coverage

**Why:** the logs reconstruct the conversation flow. We need to know the volume,
the number of distinct sessions covered, and the average tokens per session.

In [ ]:
q(f'''
SELECT COUNT(*)                                          AS total_log_rows,
       COUNT(DISTINCT SESSION_ID)                       AS distinct_log_sessions,
       MIN(MOMENT)                                      AS earliest_log,
       MAX(MOMENT)                                      AS latest_log,
       ROUND(COUNT(*) / COUNT(DISTINCT SESSION_ID), 1)   AS avg_tokens_per_session
FROM {LOG}''')

**Key insight:** the log table is roughly an order of magnitude larger than the
session table (each session has many log tokens). The avg tokens/session sets a
shape expectation for the flow-reconstruction step.

## 4 - Join integrity (sessions <-> logs)

**Why:** the analysis joins the two tables on `SESSION_ID`. We need to confirm
the join is clean: log sessions exist in sessions, and most sessions have logs.

In [ ]:
q(f'''
WITH log_ids  AS (SELECT DISTINCT SESSION_ID FROM {LOG}),
     sess_ids AS (SELECT DISTINCT SESSION_ID FROM {SES})
SELECT (SELECT COUNT(*) FROM log_ids l JOIN sess_ids s USING(SESSION_ID))   AS log_sessions_matched,
       (SELECT COUNT(*) FROM log_ids l LEFT JOIN sess_ids s USING(SESSION_ID) WHERE s.SESSION_ID IS NULL) AS log_sessions_unmatched,
       (SELECT COUNT(*) FROM sess_ids s LEFT JOIN log_ids l USING(SESSION_ID) WHERE l.SESSION_ID IS NULL) AS sessions_without_logs
''')

**Key insight:** the join is clean - log sessions match sessions, and the share
of sessions without logs is small. We can safely use a regular inner/left join
in the downstream pipeline.

## 5 - Date quality check

**Why:** during early analysis we noticed `null` and `1900-01-01` dates in the
raw data. This step quantifies them so we can apply a clean analysis window.

In [ ]:
date_q = q(f'''
SELECT CASE WHEN START_MOMENT IS NULL                                 THEN 'null'
            WHEN DATE(START_MOMENT) < DATE "2010-01-01"               THEN 'pre_2010_junk'
            WHEN DATE(START_MOMENT) BETWEEN DATE "2024-01-01" AND DATE "2025-12-31" THEN 'in_window'
            ELSE 'other' END AS bucket,
       COUNT(*) AS row_count
FROM {SES} GROUP BY bucket ORDER BY row_count DESC''')
date_q

In [ ]:
# Visualise the date-quality split
fig,ax = plt.subplots(figsize=(9,3.2))
order = ['in_window','other','pre_2010_junk','null']
d = date_q.set_index('bucket').reindex([b for b in order if b in date_q.bucket.values])
colors = {'in_window':'#009900','other':'#7E8083','pre_2010_junk':'#FBA600','null':'#E60000'}
bars = ax.barh(d.index[::-1], d['row_count'][::-1], color=[colors.get(b,'#0077C8') for b in d.index[::-1]])
for b in bars:
    ax.annotate(f'{int(b.get_width()):,}', (b.get_width(), b.get_y()+b.get_height()/2),
                xytext=(4,0), textcoords='offset points', va='center', fontweight='bold')
ax.set_title('Sessions by date-quality bucket'); ax.set_xlabel('rows'); plt.tight_layout(); plt.show()

**Key insight:** a clear minority of rows fall outside a usable window (some
`null`, some `1900-01-01` placeholders). **Decision:** restrict downstream
analysis to `2024-01-01` to `2025-12-31` so denominators reflect real activity.

## 6 - Channel distribution (entry point)

**Why:** the channel is one of the entry-point drivers in the business question.
We need to know the share each channel has and spot any tiny channels that may
need filtering for stability.

In [ ]:
ch = q(f'''
SELECT CHANNEL, COUNT(*) AS n,
       ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 2) AS pct
FROM {SES} GROUP BY CHANNEL ORDER BY n DESC''')
ch

In [ ]:
fig,ax=plt.subplots(figsize=(11,3.6))
top = ch.head(10)
ax.barh(top.CHANNEL[::-1], top.n[::-1], color='#0077C8')
ax.set_title('Top channels by session volume'); plt.tight_layout(); plt.show()

**Key insight:** a few channels (typically `voice`, `web`, `app`) carry the bulk
of traffic. The long tail of tiny channels produces unstable rates, so the
dashboard later filters to channels with >=5,000 technical sessions.

## 7 - `service_type` (Mobile vs Fixed)

**Why:** business splits TOBi sessions into Mobile (M) and Fixed (F). Confirming
the mix matters for any segment-level reading.

In [ ]:
q(f'SELECT service_type, COUNT(*) AS n FROM {SES} GROUP BY service_type ORDER BY n DESC')

**Key insight:** `service_type` is a clean two-value field (M/F) with some null.
We won't cut the analysis by it directly - the `T_` tag's client-type suffix
(`CPOS`/`CPRE`/`CFIXO`) gives finer-grained segmentation downstream.

## 8 - Intent detection: `FIRST_INTENT` and `CONFIDENCE_LEVEL`

**Why:** these are the bot's read on each session. We want to:
- see the most common intents,
- understand the **distribution of `CONFIDENCE_LEVEL`** (stored as string but numeric).

In [ ]:
q(f'''SELECT FIRST_INTENT, COUNT(*) AS n FROM {SES}
GROUP BY FIRST_INTENT ORDER BY n DESC LIMIT 25''')

In [ ]:
# CONFIDENCE_LEVEL bucket: numeric string -> none / low / medium / high
conf_q = q(f'''
WITH base AS (
  SELECT CASE
    WHEN CONFIDENCE_LEVEL IS NULL OR TRIM(CONFIDENCE_LEVEL)='' THEN 'unknown'
    WHEN SAFE_CAST(CONFIDENCE_LEVEL AS FLOAT64) IS NULL        THEN 'unknown'
    WHEN SAFE_CAST(CONFIDENCE_LEVEL AS FLOAT64) = 0            THEN 'none'
    WHEN SAFE_CAST(CONFIDENCE_LEVEL AS FLOAT64) < 0.5           THEN 'low'
    WHEN SAFE_CAST(CONFIDENCE_LEVEL AS FLOAT64) < 0.8           THEN 'medium'
    ELSE 'high' END AS confidence_band
  FROM {SES}
)
SELECT confidence_band, COUNT(*) AS n,
       ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 1) AS pct
FROM base GROUP BY confidence_band ORDER BY n DESC''')
conf_q

In [ ]:
# Visualise confidence-band split
order = ['none','unknown','low','medium','high']
c = conf_q.set_index('confidence_band').reindex([b for b in order if b in conf_q.confidence_band.values])
fig,ax=plt.subplots(figsize=(9,3.2))
colors={'none':'#E60000','unknown':'#7E8083','low':'#FBA600','medium':'#0077C8','high':'#009900'}
bars=ax.bar(c.index, c['pct'], color=[colors.get(b,'#0077C8') for b in c.index])
for b in bars: ax.annotate(f'{b.get_height():.0f}%',(b.get_x()+b.get_width()/2,b.get_height()),
                           xytext=(0,3),textcoords='offset points',ha='center',fontweight='bold')
ax.set_title('CONFIDENCE_LEVEL distribution (banded)'); ax.set_ylabel('%')
plt.tight_layout(); plt.show()

**Key insight:** confidence is **dominated by `none` (value 0) and `unknown`** -
the field is sparsely populated, so it cannot be relied on as a routing-decision
driver in the analysis. The dashboard explicitly notes this; we use topic and
channel instead.

## 9 - `IS_FUNCTIONAL` (resolution / containment signal)

**Why:** this drives the FCR (first-contact resolution) metric. Confirm domain
values (`Yes`/`No`/blank/null).

In [ ]:
fnc_q = q(f'SELECT IS_FUNCTIONAL, COUNT(*) AS n FROM {SES} GROUP BY IS_FUNCTIONAL ORDER BY n DESC')
fnc_q

In [ ]:
# Visualise IS_FUNCTIONAL distribution
f = fnc_q.copy()
f['IS_FUNCTIONAL'] = f['IS_FUNCTIONAL'].fillna('(null)')
fig,ax=plt.subplots(figsize=(9,3.2))
colors=[('#009900' if v=='Yes' else ('#E60000' if v=='No' else '#7E8083')) for v in f['IS_FUNCTIONAL']]
bars=ax.bar(f['IS_FUNCTIONAL'], f['n'], color=colors)
for b in bars: ax.annotate(f'{int(b.get_height()):,}',(b.get_x()+b.get_width()/2,b.get_height()),
                           xytext=(0,3),textcoords='offset points',ha='center',fontweight='bold')
ax.set_title('IS_FUNCTIONAL distribution'); plt.tight_layout(); plt.show()

**Key insight:** `IS_FUNCTIONAL` is essentially `Yes` / `No` with a notable share
of blank/null. **Decision:** treat `Yes` as functional/contained for FCR;
blank/null counts as unknown (i.e. *not* claimed as resolved).

## 10 - Customer dimensions (`CUSTOMER_TYPE`, `SERVICE_STATUS`)

**Why:** segments matter to leadership (high-value vs Business). We confirm the
domain values present in the data so any segment cut is grounded.

In [ ]:
q(f'''SELECT CUSTOMER_TYPE, SERVICE_STATUS, COUNT(*) AS n
FROM {SES} GROUP BY CUSTOMER_TYPE, SERVICE_STATUS
ORDER BY n DESC LIMIT 20''')

**Key insight:** the customer dimensions confirm we have a usable population mix
across `Consumer` / `Business` and service-status values - enough for the
segment-level chart in the dashboard.

## 11 - LOG token vocabulary

Each `LOG` row contains one token; the analysis reconstructs the trail by
concatenating tokens in `ROW_ID` order. The grammar is:
`S_`=Start (carries entity `E#` and intent `I#`), `R_`=Root, `M_`=Module,
`T_`=Tag, `E_`=End. `T_<digit><letter><roman>_<client>` is the routing/outcome.

**Why this matters:** the tag vocabulary determines how we classify sessions.
Below we count token types and show the top `T_` destinations.

In [ ]:
q(f'''
SELECT CASE
  WHEN STARTS_WITH(TRIM(LOG), 'S_') THEN 'S_state'
  WHEN STARTS_WITH(TRIM(LOG), 'R_') THEN 'R_root'
  WHEN STARTS_WITH(TRIM(LOG), 'M_') THEN 'M_module'
  WHEN STARTS_WITH(TRIM(LOG), 'T_') THEN 'T_tag'
  WHEN STARTS_WITH(TRIM(LOG), 'E_') THEN 'E_end'
  ELSE CONCAT('other:', SUBSTR(TRIM(LOG), 1, 2)) END AS token_type,
       COUNT(*) AS n
FROM {LOG} GROUP BY token_type ORDER BY n DESC''')

In [ ]:
# Top T_ routing destinations (the spine of the routing analysis)
q(f'''
SELECT TRIM(LOG) AS routing_target, COUNT(*) AS n_events,
       COUNT(DISTINCT SESSION_ID) AS n_sessions
FROM {LOG} WHERE STARTS_WITH(TRIM(LOG), 'T_')
GROUP BY routing_target ORDER BY n_sessions DESC LIMIT 20''')

In [ ]:
# Sample full reconstructed trails (eyeball the conversation flow)
q(f'''
SELECT SESSION_ID, COUNT(*) AS n_tokens,
       STRING_AGG(TRIM(LOG), ' > ' ORDER BY ROW_ID) AS flow_trail
FROM {LOG} GROUP BY SESSION_ID LIMIT 5''')

**Key insight:** the `T_` vocabulary in the data matches the published business
mapping ([`docs/tag_mappings.md`](../docs/tag_mappings.md)) - outcome digit
(`1`/`2`), letter (channel / outcome group), Roman numeral (support type), and
client-type suffix. This is the single most important EDA finding because it
justifies how we decode tags into outcome, support type and client segment for
the rest of the analysis.

## 12 - Null rates on key fields

**Why:** any field used downstream needs its missingness understood, so we know
where the analysis must be defensive.

In [ ]:
nulls = q(f'''
SELECT
  COUNT(*) AS row_count,
  ROUND(100*COUNTIF(FIRST_INTENT IS NULL OR FIRST_INTENT='')/COUNT(*),2) AS pct_first_intent_null,
  ROUND(100*COUNTIF(CONFIDENCE_LEVEL IS NULL OR CONFIDENCE_LEVEL='')/COUNT(*),2) AS pct_confidence_null,
  ROUND(100*COUNTIF(CHANNEL IS NULL OR CHANNEL='')/COUNT(*),2) AS pct_channel_null,
  ROUND(100*COUNTIF(IS_FUNCTIONAL IS NULL OR IS_FUNCTIONAL='')/COUNT(*),2) AS pct_is_functional_null,
  ROUND(100*COUNTIF(service_type IS NULL OR service_type='')/COUNT(*),2) AS pct_service_type_null,
  ROUND(100*COUNTIF(ANI IS NULL OR ANI='')/COUNT(*),2) AS pct_ani_null
FROM {SES}''')
nulls

In [ ]:
# Visualise null rates as a horizontal bar
n = nulls.iloc[0].drop('row_count')
n.index = [i.replace('pct_','').replace('_null','').replace('_',' ') for i in n.index]
n = n.sort_values(ascending=True)
fig,ax=plt.subplots(figsize=(9,3.6))
colors=['#E60000' if v>=20 else ('#FBA600' if v>=5 else '#009900') for v in n]
bars=ax.barh(n.index, n.values, color=colors)
for b in bars: ax.annotate(f'{b.get_width():.1f}%',(b.get_width(),b.get_y()+b.get_height()/2),
                           xytext=(4,0),textcoords='offset points',va='center',fontweight='bold')
ax.set_title('Null / blank rates on key fields'); ax.set_xlabel('% null')
plt.tight_layout(); plt.show()

**Key insight:** core fields like `CHANNEL` and `FIRST_INTENT` are well-populated;
`CONFIDENCE_LEVEL` is the most concerning (corroborates Section 8); `ANI` has
some missingness so the same-customer repeat-contact metric is computed only
where `ANI` is present.

## 13 - EDA findings & decisions

**Confirmed findings:**

1. **Volumes:** sessions in the tens of millions; logs ~10x larger than the session table.
2. **Date quality:** raw data contains `null` and `1900-01-01` rows.
   *Decision:* apply a 2024-2025 analysis window.
3. **Join integrity:** logs join cleanly to sessions; nearly all sessions have logs.
4. **`CONFIDENCE_LEVEL` is numeric stored as string**, dominated by `0`/blank.
   *Decision:* bucket into bands (`none`/`low`/`medium`/`high`/`unknown`); not used
   as a primary driver in the dashboard.
5. **`IS_FUNCTIONAL`** is `Yes` / `No` / blank / null.
   *Decision:* treat `Yes` as functional/contained for FCR.
6. **`service_type`** is `M` (Mobile) / `F` (Fixed) with some null - useful context
   only; the `T_` client-type suffix gives finer-grained segmentation.
7. **`T_` vocabulary:** matches the published business mapping
   ([`docs/tag_mappings.md`](../docs/tag_mappings.md)).
   *Decision:* decode digit (`1`=Contained, `2`=Transferred), letter (A=Bot,
   B=Digital deflection, C=Assisted deflection, D=Abandon, E=Error,
   F=Service-change, 2A=Livechat, 2B=ACD), and Roman = support type
   (`I`=Non-tech, `II`=Tech, `III`=Commercial).
8. **Technical-topic detection from entity ids (`E#`)** - more accurate than text
   keyword matching. The exact entity list is documented in `docs/tag_mappings.md`.

## 13b - OPEN ITEMS (need data-science lead's confirmation)

> **Action requested:** these four judgment calls were made provisionally based
> on the published mapping and analytical intuition. Each could change the
> downstream classification, so we want explicit business sign-off.

| # | Open item | Current handling | Why it matters |
|---|---|---|---|
| 1 | Is `general_difficulty` (intent `I8`, 'Dificuldades') genuinely a **technical** topic? | Treated as technical | If not, ~28% of currently-technical sessions move out, shifting totals (but not the conclusion that vague faults leak most). |
| 2 | Treatment of `IV/V/VI/VII` Roman-numeral tags | Mapped to `other_review` (excluded from misroute counting) | Some may be legitimate tech/non-tech queues; misclassifying them would over- or under-state misroute. |
| 3 | Should `Commercial (III)` count as a misroute for a technical topic? | Counted as a misroute | If business intent is to allow commercial cross-sell from a tech session, this should NOT count as misroute. |
| 4 | Is the technical-entity list right for **Business** sessions specifically? | Uses the same `E#` list as Consumer | Business may use different entities or troubleshooting flows that we're missing. |

## 14 - Numeric distributions: session duration & tokens-per-session

**Why (lead's feedback):** the EDA so far described categorical fields. The
lead asked for **more statistical depth** (median, percentiles, range) so we
can describe the **shape** of numeric variables - this strengthens later
claims about handling time and conversation complexity.

**What we compute:** count, mean, median, std, p25/p50/p75/p90/p95/p99 and
min/max for session duration (seconds) and log tokens per session.

In [ ]:
dur = q(f'''
WITH base AS (
  SELECT TIMESTAMP_DIFF(END_MOMENT, START_MOMENT, SECOND) AS dur_s
  FROM {SES}
  WHERE START_MOMENT IS NOT NULL AND END_MOMENT IS NOT NULL
    AND DATE(START_MOMENT) BETWEEN '2024-01-01' AND '2025-12-31'
    AND TIMESTAMP_DIFF(END_MOMENT, START_MOMENT, SECOND) BETWEEN 0 AND 86400
)
SELECT COUNT(*) AS n,
       ROUND(AVG(dur_s),1) AS mean_s,
       APPROX_QUANTILES(dur_s,100)[OFFSET(25)] AS p25,
       APPROX_QUANTILES(dur_s,100)[OFFSET(50)] AS median,
       APPROX_QUANTILES(dur_s,100)[OFFSET(75)] AS p75,
       APPROX_QUANTILES(dur_s,100)[OFFSET(90)] AS p90,
       APPROX_QUANTILES(dur_s,100)[OFFSET(95)] AS p95,
       APPROX_QUANTILES(dur_s,100)[OFFSET(99)] AS p99,
       ROUND(STDDEV(dur_s),1) AS stddev,
       MIN(dur_s) AS min_s, MAX(dur_s) AS max_s
FROM base''')
dur

In [ ]:
tok = q(f'''
WITH per_session AS (
  SELECT SESSION_ID, COUNT(*) AS n_tokens FROM {LOG} GROUP BY SESSION_ID
)
SELECT COUNT(*) AS n,
       ROUND(AVG(n_tokens),1) AS mean,
       APPROX_QUANTILES(n_tokens,100)[OFFSET(25)] AS p25,
       APPROX_QUANTILES(n_tokens,100)[OFFSET(50)] AS median,
       APPROX_QUANTILES(n_tokens,100)[OFFSET(75)] AS p75,
       APPROX_QUANTILES(n_tokens,100)[OFFSET(90)] AS p90,
       APPROX_QUANTILES(n_tokens,100)[OFFSET(95)] AS p95,
       APPROX_QUANTILES(n_tokens,100)[OFFSET(99)] AS p99,
       ROUND(STDDEV(n_tokens),1) AS stddev,
       MIN(n_tokens) AS min, MAX(n_tokens) AS max
FROM per_session''')
tok

In [ ]:
hist = q(f'''
WITH base AS (
  SELECT TIMESTAMP_DIFF(END_MOMENT, START_MOMENT, SECOND) AS dur_s
  FROM {SES}
  WHERE START_MOMENT IS NOT NULL AND END_MOMENT IS NOT NULL
    AND DATE(START_MOMENT) BETWEEN '2024-01-01' AND '2025-12-31'
    AND TIMESTAMP_DIFF(END_MOMENT, START_MOMENT, SECOND) BETWEEN 0 AND 1800
)
SELECT CAST(FLOOR(dur_s/30) AS INT64)*30 AS bucket_s, COUNT(*) AS sessions
FROM base GROUP BY bucket_s ORDER BY bucket_s''')
fig,ax=plt.subplots(figsize=(11,3.6))
ax.bar(hist.bucket_s, hist.sessions, width=28, color='#0077C8')
ax.set_title('Session duration histogram (30-second buckets, 0-30 min)')
ax.set_xlabel('seconds'); ax.set_ylabel('sessions'); plt.tight_layout(); plt.show()

**Key insight:** session durations are **right-skewed** - most sessions
resolve in well under a minute, but a long tail of long sessions pulls the
mean above the median. **For comparing cohorts, prefer the median over the
mean** to avoid letting outliers distort findings.

## 15 - Internal-session investigation (per Sonia's request)

**Why:** the data-science lead noted `INTERNAL_SES_LIST` is only populated for
certain channels (likely **App** and **Web**) and asked for explicit
validation. This determines where the field is informative for impact analysis.

**What we check:**
1. `INTERNAL_SES_LIST` coverage rate by channel.
2. Distribution of internal sessions listed per parent.
3. `NEXT_SESSION_ID` coverage rate by channel (for symmetry).

In [ ]:
iss = q(f'''
SELECT CHANNEL, COUNT(*) AS sessions,
       ROUND(100*COUNTIF(INTERNAL_SES_LIST IS NOT NULL AND INTERNAL_SES_LIST!='')/COUNT(*),2) AS pct_with_internal,
       ROUND(100*COUNTIF(NEXT_SESSION_ID IS NOT NULL AND NEXT_SESSION_ID!='')/COUNT(*),2)       AS pct_with_next
FROM {SES}
WHERE DATE(START_MOMENT) BETWEEN '2024-01-01' AND '2025-12-31'
GROUP BY CHANNEL ORDER BY sessions DESC''')
iss

In [ ]:
top = iss[iss.sessions >= 50000].sort_values('pct_with_internal', ascending=True)
fig,ax = plt.subplots(figsize=(11,4))
bars = ax.barh(top.CHANNEL, top.pct_with_internal, color='#0077C8')
for b in bars: ax.annotate(f'{b.get_width():.1f}%',(b.get_width(),b.get_y()+b.get_height()/2),
                            xytext=(4,0),textcoords='offset points',va='center',fontweight='bold')
ax.set_title('INTERNAL_SES_LIST coverage by channel  (channels with >=50k sessions)')
ax.set_xlabel('% of sessions with INTERNAL_SES_LIST populated'); ax.margins(x=0.2)
plt.tight_layout(); plt.show()

In [ ]:
isd = q(f'''
WITH base AS (
  SELECT ARRAY_LENGTH(SPLIT(INTERNAL_SES_LIST, ',')) AS n_internal
  FROM {SES}
  WHERE INTERNAL_SES_LIST IS NOT NULL AND INTERNAL_SES_LIST != ''
    AND DATE(START_MOMENT) BETWEEN '2024-01-01' AND '2025-12-31'
)
SELECT COUNT(*) AS parents,
       ROUND(AVG(n_internal),2) AS mean_internal,
       APPROX_QUANTILES(n_internal,100)[OFFSET(50)] AS median_internal,
       APPROX_QUANTILES(n_internal,100)[OFFSET(75)] AS p75,
       APPROX_QUANTILES(n_internal,100)[OFFSET(95)] AS p95,
       MIN(n_internal) AS min, MAX(n_internal) AS max
FROM base''')
isd

**Key insight:** confirm visually which channels populate `INTERNAL_SES_LIST`
(expect `app` and `web` per Sonia). Where coverage is near 0%, any analysis
relying on the field is uninformative - so misroute impact should be derived
primarily from `NEXT_SESSION_ID` + same-`ANI` repeats, with `INTERNAL_SES_LIST`
adding signal only on the digital channels.

> **Open item (from meeting minutes):** Sonia noted a `CD` table with a
> `SESSION_ID` that, after cleaning, can be joined to the internal session
> list. Confirm the full BigQuery path so we can run a join-coverage test.

## 16 - Customer-journey depth (sessions per customer)

**Why:** Step 3 of the framework includes customer journeys. We need to know
how often a customer contacts TOBi to ground repeat-contact and journey-impact
arguments.

In [ ]:
spa = q(f'''
WITH per_ani AS (
  SELECT ANI, COUNT(*) AS sessions
  FROM {SES}
  WHERE ANI IS NOT NULL AND ANI != ''
    AND DATE(START_MOMENT) BETWEEN '2024-01-01' AND '2025-12-31'
  GROUP BY ANI
)
SELECT COUNT(*) AS distinct_anis,
       ROUND(AVG(sessions),2) AS mean_sessions_per_ani,
       APPROX_QUANTILES(sessions,100)[OFFSET(50)] AS median,
       APPROX_QUANTILES(sessions,100)[OFFSET(75)] AS p75,
       APPROX_QUANTILES(sessions,100)[OFFSET(90)] AS p90,
       APPROX_QUANTILES(sessions,100)[OFFSET(95)] AS p95,
       APPROX_QUANTILES(sessions,100)[OFFSET(99)] AS p99,
       MAX(sessions) AS max_sessions
FROM per_ani''')
spa

In [ ]:
spa_hist = q(f'''
WITH per_ani AS (
  SELECT ANI, COUNT(*) AS sessions
  FROM {SES}
  WHERE ANI IS NOT NULL AND ANI != ''
    AND DATE(START_MOMENT) BETWEEN '2024-01-01' AND '2025-12-31'
  GROUP BY ANI
)
SELECT LEAST(sessions, 20) AS contacts, COUNT(*) AS customers
FROM per_ani GROUP BY contacts ORDER BY contacts''')
fig,ax=plt.subplots(figsize=(10,3.6))
ax.bar(spa_hist.contacts.astype(str), spa_hist.customers, color='#0077C8')
ax.set_title('Sessions-per-customer distribution  (capped at 20+)')
ax.set_xlabel('contacts per customer'); ax.set_ylabel('customers')
plt.tight_layout(); plt.show()

**Key insight:** most customers have only a few contacts in the window; a
small tail of high-frequency customers drives a disproportionate volume share.
The repeat-contact rate in the dashboard is **per session**, so a few power
users do not distort the conclusion.

## 17 - Temporal patterns (hour-of-day, day-of-week)

**Why:** the dashboard heatmap shows *misroutes* by hour x day. Here we show
**all sessions** so we can tell whether misroute peaks line up with overall
contact peaks (volume effect) or stand out (rate effect).

In [ ]:
t = q(f'''
SELECT EXTRACT(DAYOFWEEK FROM START_MOMENT) AS dow,
       EXTRACT(HOUR FROM START_MOMENT) AS hour,
       COUNT(*) AS sessions
FROM {SES}
WHERE START_MOMENT IS NOT NULL
  AND DATE(START_MOMENT) BETWEEN '2024-01-01' AND '2025-12-31'
GROUP BY dow, hour''')
piv = t.pivot_table(index='dow', columns='hour', values='sessions', fill_value=0)
fig,ax=plt.subplots(figsize=(13,3.6))
im = ax.imshow(piv.values, aspect='auto', cmap='Blues')
ax.set_yticks(range(7)); ax.set_yticklabels(['Sun','Mon','Tue','Wed','Thu','Fri','Sat'])
ax.set_xticks(range(24)); ax.set_xticklabels(piv.columns)
ax.set_title('All TOBi sessions by day-of-week x hour-of-day'); ax.set_xlabel('hour')
fig.colorbar(im, ax=ax, shrink=.8, label='sessions'); plt.tight_layout(); plt.show()

**Key insight:** TOBi volume concentrates on weekday daytime hours. Compare
to the dashboard misroute heatmap: if peaks share the same pattern it is a
volume effect; if misroutes peak at *different* times, it is a quality issue.

## 18 - Routing & topic vocabulary depth

**Why:** describe the shape of the `T_` tag vocabulary at token level and the
entity codes feeding the technical-topic detection.

In [ ]:
ntg = q(f'''
WITH per_session AS (
  SELECT SESSION_ID, COUNTIF(STARTS_WITH(TRIM(LOG),'T_')) AS n_tags
  FROM {LOG} GROUP BY SESSION_ID
)
SELECT COUNT(*) AS n,
       ROUND(AVG(n_tags),2) AS mean,
       APPROX_QUANTILES(n_tags,100)[OFFSET(50)] AS median,
       APPROX_QUANTILES(n_tags,100)[OFFSET(75)] AS p75,
       APPROX_QUANTILES(n_tags,100)[OFFSET(90)] AS p90,
       APPROX_QUANTILES(n_tags,100)[OFFSET(95)] AS p95,
       MAX(n_tags) AS max_tags
FROM per_session''')
ntg

In [ ]:
out = q(f'''
SELECT REGEXP_EXTRACT(TRIM(LOG), r'^T_([12])') AS outcome_digit,
       COUNT(*) AS tokens, COUNT(DISTINCT SESSION_ID) AS sessions
FROM {LOG}
WHERE STARTS_WITH(TRIM(LOG),'T_')
GROUP BY outcome_digit ORDER BY tokens DESC''')
out

In [ ]:
ent = q(f'''
SELECT SAFE_CAST(REGEXP_EXTRACT(TRIM(LOG), r'_E([0-9]+)') AS INT64) AS entity_id,
       COUNT(DISTINCT SESSION_ID) AS sessions
FROM {LOG}
WHERE STARTS_WITH(TRIM(LOG),'S_')
GROUP BY entity_id HAVING entity_id IS NOT NULL
ORDER BY sessions DESC LIMIT 25''')
ent

**Key insight:** the `T_` tag distribution is concentrated on a small number
of high-volume destinations - so fixing a handful of routing rules moves
large numbers of sessions. Top entity codes line up with the technical-topic
list in `docs/tag_mappings.md`, validating the topic-detection approach.

## 19 - Mapping to the 6-step framework

Per the data-science lead's framework:

| Step | Description | Where it lives |
|---|---|---|
| 1. Data Understanding | Tables, joins, business context, key metrics | `01_eda.ipynb` Sections 1-4, 11; `docs/tag_mappings.md` |
| 2. Data Validation | Quality, completeness, consistency, reliability | `01_eda.ipynb` Sections 5, 12 |
| 3. EDA | Distributions, trends, routing/escalation patterns, customer journeys | `01_eda.ipynb` Sections 6-10, **14-18 (new)** |
| 4. Insight Validation | Validate hypotheses, confirm root causes, quantify impact | `02_executive_dashboard.ipynb` Steps 5-8, 12 |
| 5. Business Analysis | CX, resolution time, transfers, operational load | `02_executive_dashboard.ipynb` Steps 4, 7, 9-11 |
| 6. Recommendations | Routing / escalation / flow improvements, expected benefits | `02_executive_dashboard.ipynb` Step 12 + `docs/recommendations.md` |

## 19b - OPEN ITEMS (extended, post-meeting)

From `13b` plus new items from the deeper EDA and meeting minutes. Items now
**resolved by the data-science lead** are struck through.

| # | Open item | Status |
|---|---|---|
| 1 | ~~Is `general_difficulty` (intent `I8`) genuinely a **technical** topic?~~ | **Resolved:** classification driven by the **last `S_` token only** (see §19c). |
| 2 | Treatment of `IV/V/VI/VII` Roman-numeral tags | Open |
| 3 | Should `Commercial (III)` count as a misroute for a technical topic? | Open |
| 4 | Is the technical-entity list right for **Business** sessions specifically? | Open |
| 5 | Confirm `service_type` codes (`M` = Mobile, `F` = Fixed)? | Open |
| 6 | Confirm `SERVICE_STATUS` codes (`AC`, `DE`, `HL`...) | Open |
| 7 | Confirm `NEXT_SESSION_ID` semantics - continuation/repeat? | Open |
| 8 | Confirm `INTERNAL_SES_LIST` semantics - intra-session handovers? | Open |
| 9 | ~~**`CD` table** with `SESSION_ID` (per Sonia)~~ | **Resolved:** confirmed as **`ACD`** table; awaiting the BigQuery path from the lead. |

## 19c - LEAD'S CORRECTION (applied) - technical topic from the LAST `S_` only

**What changed:** the technical-topic classifier now extracts the `E#` (entity)
and `I#` (intent) codes from the **last `S_` token in the session** (ordered by
`ROW_ID`), not from all `S_` tokens.

**Why this matters:** the *last* `S_` is the bot's final understanding of what
the customer wants. Earlier `S_` tokens may reflect mid-conversation pivots and
would otherwise over-classify sessions as technical.

**Where it's applied:** `standalone/session_master_query.sql` (CTEs `last_s`,
`entities`, `topic_flags`). `docs/data_cleaning.md` rule #10 documents this.

**Expected impact on the numbers:** the technical population shrinks vs the
previous run. The misrouting **pattern** (vague intents leak to non-technical
queues) should remain — but the absolute KPIs need to be **regenerated**:
rebuild `session_master` (Step 3 of `02_executive_dashboard.ipynb`) then refresh
the charts.

**Next ACD table investigation:** the `ACD` table from the lead will let us
join cleaned `SESSION_ID` to `INTERNAL_SES_LIST` for the App / Web channels.
Section 15 of this notebook already validates which channels populate
`INTERNAL_SES_LIST`, so it's a small extension once we have the table path.

---

**Next step:** see [`docs/data_cleaning.md`](../docs/data_cleaning.md) for the
cleaning rules these findings drive, and [`docs/analysis_framework.md`](../docs/analysis_framework.md)
for how every artefact maps to the 6 steps. Then
[`02_executive_dashboard.ipynb`](02_executive_dashboard.ipynb) carries Steps 4-6.